After the EDA, we left some decisions to the preprocessing stage. Here it is a summary
---

**FEATURES TO DROP**
- P_emaildomain, R_emaildomain — no clear fraud signal
- C3 — fraud rate practically 0 across all bins
- D6, D7, D8, D9, D12, D13, D14 — 87%+ nulls
- V138-V166, V167-V216, V217-V278, V322-V339 — 76-86% nulls
- V features with correlation > 0.9 within each null group
- id_07, id_08, id_21, id_22, id_23, id_24, id_25, id_26, id_27 — 95%+ nulls
- id_30 — duplicates DeviceOS information with fewer entries
- DeviceInfo — replaced by DeviceOS

**FEATURES TO TRANSFORM**
- card4 → one-hot (5 categories, drop_first model-dependent)
- card6 → one-hot (debit, credit, other, drop_first model-dependent)
- ProductCD → one-hot if not using XGBoost
- DeviceType → one-hot
- id_31 → extract browser name as new feature (DeviceBrowser)
- TransactionAmt → evaluate log-transform due to heavy right skew (model-dependent)
- TransactionDT → convert to days

**FEATURES TO CREATE**
- has_identity — binary, 7.8% vs 2.1% fraud rate
- DeviceOS — grouped from DeviceInfo values
- DeviceBrowser — extracted from id_31

**PENDING DECISIONS**
- D2, D3, D5, D11 — clear signal but ~50% nulls, decide based on causality analysis
- M1-M9 → transform into two boolean columns (M_known + M_value)
- Chronological split: 60/20/20 → 109 days / 36 days / 36 days

In [107]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import importlib
import preprocessing


In [108]:
df_merged = pd.read_parquet('../data/merged.parquet')
df_merged.shape


(590540, 438)

**FEATURES TO DROP**

In [109]:
df_merged.drop(columns=[
    # No clear fraud signal
    'P_emaildomain', 'R_emaildomain',
    # Near-zero fraud rate
    'C3',
    # 87%+ nulls
    'D6', 'D7', 'D8', 'D9', 'D12', 'D13', 'D14',
    # 76-86% nulls
    'V138', 'V139', 'V140', 'V141', 'V142', 'V143', 'V144', 'V145', 'V146', 'V147',
    'V148', 'V149', 'V150', 'V151', 'V152', 'V153', 'V154', 'V155', 'V156', 'V157',
    'V158', 'V159', 'V160', 'V161', 'V162', 'V163', 'V164', 'V165', 'V166',
    'V167', 'V168', 'V169', 'V170', 'V171', 'V172', 'V173', 'V174', 'V175', 'V176',
    'V177', 'V178', 'V179', 'V180', 'V181', 'V182', 'V183', 'V184', 'V185', 'V186',
    'V187', 'V188', 'V189', 'V190', 'V191', 'V192', 'V193', 'V194', 'V195', 'V196',
    'V197', 'V198', 'V199', 'V200', 'V201', 'V202', 'V203', 'V204', 'V205', 'V206',
    'V207', 'V208', 'V209', 'V210', 'V211', 'V212', 'V213', 'V214', 'V215', 'V216',
    'V217', 'V218', 'V219', 'V220', 'V221', 'V222', 'V223', 'V224', 'V225', 'V226',
    'V227', 'V228', 'V229', 'V230', 'V231', 'V232', 'V233', 'V234', 'V235', 'V236',
    'V237', 'V238', 'V239', 'V240', 'V241', 'V242', 'V243', 'V244', 'V245', 'V246',
    'V247', 'V248', 'V249', 'V250', 'V251', 'V252', 'V253', 'V254', 'V255', 'V256',
    'V257', 'V258', 'V259', 'V260', 'V261', 'V262', 'V263', 'V264', 'V265', 'V266',
    'V267', 'V268', 'V269', 'V270', 'V271', 'V272', 'V273', 'V274', 'V275', 'V276',
    'V277', 'V278',
    'V322', 'V323', 'V324', 'V325', 'V326', 'V327', 'V328', 'V329', 'V330', 'V331',
    'V332', 'V333', 'V334', 'V335', 'V336', 'V337', 'V338', 'V339',
    # 95%+ nulls in identity
    'id_07', 'id_08', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27',
    # Redundant with DeviceOS
    'id_30',
    # Replaced by DeviceOS
    'DeviceInfo'
], inplace=True)

In [110]:
df_merged.shape


(590540, 258)

### High correlated features to drop
From the V group of features, remove the list of features with high correlation

In [111]:
importlib.reload(preprocessing)

from preprocessing import high_corr_list


v_groups = {
    'V1-V11': list(range(1, 12)),
    'V12-V34': list(range(12, 35)),
    'V35-V52': list(range(35, 53)),
    'V53-V74': list(range(53, 75)),
    'V75-V94': list(range(75, 95)),
    'V95-V137': list(range(95, 138)),
    'V279-V321': list(range(279, 322)),
}

cols_to_drop = high_corr_list(v_groups, df_merged)
df_merged.drop(columns=list(cols_to_drop), inplace=True)

In [112]:
print(len(cols_to_drop))
print(df_merged.shape)

69
(590540, 189)


### Transform M features into boolean columns
From M1-M9 (except M4) we are going to create new columns M_Unknown and M_Value and the M is removed.

In [113]:
importlib.reload(preprocessing)
from preprocessing import encode_boolean_known_features
df_merged = encode_boolean_known_features(df_merged, ['M1', 'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9'], "T")


In [114]:
df_merged.shape

(590540, 197)

### Card4, Card6 one-hot transformation

In [115]:
importlib.reload(preprocessing)
from preprocessing import encode_card_features

df_merged = encode_card_features(df_merged, 'card4', ['visa', 'mastercard', 'american express', 'discover'], False)
df_merged.shape

(590540, 201)

In [116]:
df_merged = encode_card_features(df_merged, 'card6', ['debit', 'credit'], False)
df_merged.shape

(590540, 203)

### ProductCD and DeviceType one-hot transformation

In [117]:
df_merged = encode_card_features(df_merged, 'ProductCD',  ['W','H','C','S','R'], False)
df_merged.shape

(590540, 208)

In [118]:
df_merged = encode_card_features(df_merged, 'DeviceType',  ['desktop', 'mobile'], False)
df_merged.shape

(590540, 210)

### DeviceBrowser
Firstly we extract the browser and secondly we encode card features

In [119]:
importlib.reload(preprocessing)
from preprocessing import extract_browser
df_merged['DeviceBrowser'] = df_merged['id_31'].apply(extract_browser)
df_merged.drop(columns=['id_31'], inplace=True)

In [120]:
df_merged = encode_card_features(df_merged, 'DeviceBrowser',  ['safari', 'firefox', 'ie', 'chrome'], False)
df_merged.shape


(590540, 214)

### DeviceOS
Same as before, Firstly we extract the OS and then we encode card features

In [121]:
importlib.reload(preprocessing)
from preprocessing import extract_os
df_merged = encode_card_features(df_merged, 'DeviceOS',  ['windows', 'android', 'ios', 'mac'], False)
df_merged.shape

(590540, 218)

### TransactionDT
Remove this column because we created in the EDA the TransactionDTDays

In [122]:
df_merged.drop(columns=['TransactionDT'], inplace=True)

In [123]:
df_merged.shape

(590540, 217)

### D2, D3, D5, D11
- Clear signal — fraud rate 8-11% in low bins vs 1-2% in high bins
- ~50% nulls
- XGBoost manage nulls natively. We will decide the model to apply later.

### Notes after first training iteration
XGBoost offers `enable_categorical=True` as a native option, but after evaluation we ruled it out: it is experimental and, critically, it cannot handle unseen categories at inference time — a category present in production but absent from training would cause the model to fail.
So, the pending columns for preprocessing:

**Drop:**
- id_33 — high cardinality, screen resolution

**Transform to boolean (Found/NotFound):**
- id_12, id_28, id_16, id_29 — True if Found, False otherwise. New method encode_simple_boolean_features
- id_15 — special treatment: Found/New/NaN → two boolean columns id_15_found and id_15_value. Similar to encode_boolean_known_features

**Transform to numeric:**
- id_34 — match_status (0, 1, 2, -1)

**Transform to boolean (T/F):**

- id_35, id_36, id_37, id_38

**Needs encode_card_features:**

- M4 — categorical with values M0, M1, M2, NaN

In [124]:
df_merged.drop(columns=['id_33'], inplace=True)

In [125]:
importlib.reload(preprocessing)
from preprocessing import encode_simple_boolean_features
df_merged = encode_simple_boolean_features(df_merged, ['id_12', 'id_28', 'id_16', 'id_29'], "Found")


In [126]:
df_merged = encode_boolean_known_features(df_merged, ['id_15'], "Found")

In [127]:
importlib.reload(preprocessing)
from preprocessing import encode_match_status

df_merged = encode_match_status(df_merged)
df_merged['id_34'].value_counts(dropna=False)

id_34
 NaN    512735
 2.0     60011
 1.0     17376
 0.0       415
-1.0         3
Name: count, dtype: int64

In [128]:
importlib.reload(preprocessing)
from preprocessing import encode_tf_features

df_merged = encode_tf_features(df_merged, ['id_35', 'id_36', 'id_37', 'id_38'])
df_merged.shape

(590540, 217)

In [129]:
df_merged['M4'].value_counts(dropna=False)
df_merged = encode_card_features(df_merged, 'M4', ['M0', 'M1', 'M2'], False)

### Split dataframe into train, validation and test

In [130]:
importlib.reload(preprocessing)
from preprocessing import split_dataframe
df_train, df_validation, df_test = split_dataframe(df_merged)


In [131]:
print(df_train.shape)
print(df_validation.shape)
print(df_test.shape)

(354324, 220)
(118108, 220)
(118108, 220)


In [132]:
# Before closing the Preprocessing, we will save the three data sets in parquet format

df_train.to_parquet('../data/train.parquet')
df_validation.to_parquet('../data/validation.parquet')
df_test.to_parquet('../data/test.parquet')
